## 🏡 House Price Prediction - Advanced Regression Techniques

## Business Understanding

Predicting house prices accurately is a crucial task for real estate developers, investors, and potential homeowners. Traditional appraisals rely on a few key metrics like square footage, bedrooms, and location. However, the true value of a house is often dictated by dozens of subtle factors, ranging from the height of the basement to the condition of the garage.

In this project, we analyze the **Ames Housing dataset**, which contains 79 explanatory variables describing residential homes in Ames, Iowa. The objective is to build a robust machine learning regression model that can predict the final `SalePrice` of a home based on its characteristics.

## Dataset Overview

The dataset contains a rich set of features, including:

- **Spatial Features**: `LotArea`, `GrLivArea`, `TotalBsmtSF`, `GarageArea`
- **Quality/Condition Features**: `OverallQual`, `OverallCond`, `ExterQual`, `KitchenQual`
- **Temporal Features**: `YearBuilt`, `YearRemodAdd`, `YrSold`, `MoSold`
- **Categorical Features**: `Neighborhood`, `HouseStyle`, `BldgType`, `SaleCondition`
- **Target variable**: `SalePrice` (Continuous)

> Total rows: 1460 | Features: 80 | Domain: **Real Estate Analytics**

## Problem Type

This is a **Supervised Regression Problem**.

We will use machine learning algorithms to predict a continuous numerical value (`SalePrice`). Because house prices can vary drastically, we will log-transform the target variable to ensure that errors in predicting expensive and cheap houses affect the model equally. The primary evaluation metric will be **Root Mean Squared Error (RMSE)**.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.linear_model import Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor

from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, KFold
import shap

In [ ]:
# Load the dataset
data = pd.read_csv('C:/Users/arbaj/Projects/House Price Predictions/data/data.csv')

In [ ]:
data.head()

In [ ]:
data.info()

## Exploratory Data Analysis (EDA)

In [ ]:
# ✅ Distribution of Target Variable: SalePrice
plt.figure(figsize=(10, 5))
sns.histplot(data['SalePrice'], kde=True, bins=50, color='blue')
plt.title('Distribution of SalePrice')
plt.xlabel('Sale Price ($)')
plt.show()

💡 **Insight**:
The `SalePrice` is right-skewed. Most houses fall in the $100k - $250k range, but there are a few highly expensive outliers. We will apply a `log1p` transformation during preprocessing to normalize this distribution.

In [ ]:
# ✅ Ground Living Area vs SalePrice
plt.figure(figsize=(10, 6))
sns.scatterplot(x='GrLivArea', y='SalePrice', data=data, alpha=0.6)
plt.title('Above Ground Living Area vs. SalePrice')
plt.xlabel('Ground Living Area (sq ft)')
plt.ylabel('Sale Price ($)')
plt.show()

💡 **Insight**:
There is a strong positive linear relationship between living area and price. We can also spot a couple of massive houses (>4000 sq ft) that sold for surprisingly low prices. These are likely outliers that should be removed to prevent model skewing.

In [ ]:
# ✅ Overall Quality vs SalePrice
plt.figure(figsize=(10, 6))
sns.boxplot(x='OverallQual', y='SalePrice', data=data, palette='viridis')
plt.title('Overall Quality vs. SalePrice')
plt.xlabel('Overall Quality (1-10)')
plt.ylabel('Sale Price ($)')
plt.show()

💡 **Insight**:
`OverallQual` is arguably the most critical categorical/ordinal feature. As the material and finish quality increases, the median sale price increases exponentially.

In [ ]:
# Removing extreme outliers based on GrLivArea
data = data.drop(data[(data['GrLivArea'] > 4000) & (data['SalePrice'] < 300000)].index)
data.reset_index(drop=True, inplace=True)